# VisionGuard training and DagsHub tracking

This notebook runs the reproducible audit, split generation, training, and MLflow logging workflow in Google Colab. Add `DAGSHUB_USER` and `DAGSHUB_TOKEN` as Colab Secrets before running the tracking cell. Never place the token in notebook code or screenshots.

In [5]:
!git clone https://github.com/abdul-raheem-fast/23L-0588_A2_Ops.git
%cd 23L-0588_A2_Ops
%pip install -q -r requirements.txt kagglehub mlflow==2.19.0 keras-tuner==1.4.7
import os
import random
import numpy as np
import tensorflow as tf
print('TensorFlow:', tf.__version__)
print('GPU devices:', tf.config.list_physical_devices('GPU'))
SEED = 23
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

Cloning into '23L-0588_A2_Ops'...
remote: Enumerating objects: 83, done.
remote: Counting objects: 100% (83/83), done.
remote: Compressing objects: 100% (65/65), done.
remote: Total 83 (delta 19), reused 74 (delta 10), pack-reused 0 (from 0)
Receiving objects: 100% (83/83), 1.28 MiB | 38.60 MiB/s, done.
Resolving deltas: 100% (19/19), done.
/content/23L-0588_A2_Ops/23L-0588_A2_Ops
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done

  Preparing metadata (pyproject.toml) ... done
ERROR: Ignored the following versions that require a different python version: 1.21.2 Requires-Python >=3.7,<3.11; 1.21.3 Requires-Python >=3.7,<3.11; 1.21.4 Requires-Python >=3.7,<3.11; 1.21.5 Requires-Python >=3.7,<3.11; 1.21.6 Requires-Python >=3.7,<3.11; 1.26.0 Requires-Python <3.13,>=3.9; 1.26.1 Requires-Python <3.13,>=3.9
ERROR: Could not find a version that satisfies the requirement tensorflow-cpu==2.18.0 (from versions: 2.20.

In [6]:
import kagglehub
dataset_path = kagglehub.dataset_download('cashbowman/ai-generated-images-vs-real-images')
print('Path to dataset files:', dataset_path)
from pathlib import Path
for path in sorted(Path(dataset_path).iterdir()):
    print(path, 'files:', sum(1 for item in path.rglob('*') if item.is_file()))

Using Colab cache for faster access to the 'ai-generated-images-vs-real-images' dataset.
Path to dataset files: /kaggle/input/ai-generated-images-vs-real-images
/kaggle/input/ai-generated-images-vs-real-images/AiArtData files: 539
/kaggle/input/ai-generated-images-vs-real-images/RealArt files: 436


In [7]:
from pathlib import Path
from scripts.audit_dataset import audit
from scripts.make_splits import main as make_splits_main
import sys
audit_output = Path('artifacts/audit')
audit_output.mkdir(parents=True, exist_ok=True)
rows = audit(dataset_path)
import csv, json
with (audit_output / 'image_audit.csv').open('w', newline='', encoding='utf-8') as file:
    writer = csv.DictWriter(file, fieldnames=rows[0].keys())
    writer.writeheader()
    writer.writerows(rows)
print('Audited images:', len(rows))
sys.argv = ['make_splits', '--dataset-dir', dataset_path, '--output-dir', 'artifacts/splits', '--seed', '23']
make_splits_main()

Audited images: 975
Created reproducible train/validation/test splits in artifacts/splits


In [1]:
try:
    import pyarrow
    import mlflow
except (AttributeError, ImportError):
    print("Detected pyarrow binary incompatibility. Upgrading and restarting runtime to reload binaries...")
    %pip install -q -U pyarrow mlflow==2.19.0
    import os
    os.kill(os.getpid(), 9)

from google.colab import userdata
import os
os.environ['DAGSHUB_USER'] = userdata.get('DAGSHUB_USER')
os.environ['DAGSHUB_TOKEN'] = userdata.get('DAGSHUB_TOKEN')
os.environ['MLFLOW_TRACKING_USERNAME'] = os.environ['DAGSHUB_USER']
os.environ['MLFLOW_TRACKING_PASSWORD'] = os.environ['DAGSHUB_TOKEN']
os.environ['MLFLOW_TRACKING_URI'] = f"https://dagshub.com/{os.environ['DAGSHUB_USER']}/23L-0588_A2_Ops.mlflow"
os.environ['MLFLOW_EXPERIMENT'] = 'VisionGuard'
import mlflow
mlflow.set_tracking_uri(os.environ['MLFLOW_TRACKING_URI'])
print('DagsHub MLflow URI configured for user:', os.environ['DAGSHUB_USER'])

DagsHub MLflow URI configured for user: abdul-raheem-fast


In [8]:
!pip install -q "protobuf>=6.31.1,<7.0.0"
import json
import os
from pathlib import Path

# Force pure-Python implementation if MLflow encounters serialization descriptor conflicts
os.environ["PROTOCOL_BUFFERS_PYTHON_IMPLEMENTATION"] = "python"

# Safely locate the cloned repository directory root
repo_dir = Path('/content/23L-0588_A2_Ops')
if not repo_dir.exists():
    repo_dir = Path.cwd()

# Programmatically fix the MLflow API argument bug in scripts/train.py
train_script_path = repo_dir / 'scripts' / 'train.py'
if train_script_path.exists():
    script_content = train_script_path.read_text()
    fixed_content = script_content.replace('mlflow.tensorflow.log_model(model, name="model")', 'mlflow.tensorflow.log_model(model, artifact_path="model")')
    train_script_path.write_text(fixed_content)
    print("Patched scripts/train.py: changed name='model' to artifact_path='model'.")

config_path = repo_dir / 'configs' / 'visionguard.json'
if not config_path.exists():
    # Try looking in nested folders if any
    found_paths = list(repo_dir.rglob('configs/visionguard.json'))
    if found_paths:
        config_path = found_paths[0]

# Read and update the config
config = json.loads(config_path.read_text())
config.update({'data_dir': str(repo_dir / 'artifacts/splits'), 'epochs': 10, 'batch_size': 16, 'seed': 23})

colab_config_path = repo_dir / 'configs' / 'colab.json'
colab_config_path.write_text(json.dumps(config, indent=2))

# Run training within the repository root directory
os.chdir(repo_dir)
!python scripts/train.py --config configs/colab.json

Patched scripts/train.py: changed name='model' to artifact_path='model'.
I0000 00:00:1792608813.043629   16741 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
W0000 00:00:1792608817.733735   16741 gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was false.
I0000 00:00:1792608817.735600   16741 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
Epoch 1/10
I0000 00:00:1792608835.564900   16883 shuffle_dataset_op.cc:453] ShuffleDatasetV3:2: Filling up shuffle buffer (this may take a while): 645 of 779
I0000 00:00:1792608837.923926   16883 shuffle_datase

In [9]:
from google.colab import files
from pathlib import Path
import shutil

# Safely locate the cloned repository directory root
repo_dir = Path('/content/23L-0588_A2_Ops')
if not repo_dir.exists():
    repo_dir = Path.cwd()

candidates = list((repo_dir / 'models').glob('*.keras'))
print('Saved models:', [str(path) for path in candidates])

best = repo_dir / 'models/mobilenetv2_finetuned.keras'
if best.exists():
    target_dir = repo_dir / 'inference-service/models'
    target_dir.mkdir(parents=True, exist_ok=True)
    shutil.copy2(best, target_dir / 'champion.keras')
    print('Champion copied to inference-service/models/champion.keras')

# Create archives relative to the repository directory
shutil.make_archive('/content/visionguard-training-artifacts', 'zip', str(repo_dir / 'artifacts'))
shutil.make_archive('/content/visionguard-models', 'zip', str(repo_dir / 'models'))

# Trigger downloads
files.download('/content/visionguard-training-artifacts.zip')
files.download('/content/visionguard-models.zip')

Saved models: ['/content/23L-0588_A2_Ops/models/mobilenetv2_finetuned.keras', '/content/23L-0588_A2_Ops/models/mobilenetv2_head.keras', '/content/23L-0588_A2_Ops/models/baseline_cnn.keras']
Champion copied to inference-service/models/champion.keras


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>